In [5]:
import os
import json
import random
import re
import boto3
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# MongoDB connection details
MONGODB_URI = os.getenv('MONGODB_URI_REALM')
DB_NAME = 'gpt'
COLLECTION_NAME = 'list'

# Cloudflare R2 bucket details
R2_ACCESS_KEY = os.getenv('CLOUDFLARE_ACCESS_KEY')
R2_SECRET_KEY = os.getenv('CLOUDFLARE_SECRET_KEY')
R2_BUCKET = os.getenv('CLOUDFLARE_BUCKET')
R2_ENDPOINT = os.getenv('CLOUDFLARE_ENDPOINT')

# Slugify function to convert strings to URL-friendly slugs
def slugify(value):
    if not value:
        return None
    value = value.lower()
    value = re.sub(r'[^\w\s-]', '', value)  # Remove non-word characters
    value = re.sub(r'[\s_]+', '-', value)  # Replace spaces and underscores with hyphens
    return value

# Upload files to Cloudflare R2 bucket
def upload_to_r2(local_file_path, r2_key):
    session = boto3.session.Session()
    s3 = session.client(
        's3',
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    print(f"Uploading {local_file_path} to R2 as {r2_key}")
    if not os.path.exists(local_file_path):
        print(f"Error: {local_file_path} does not exist.")
        return
    if os.path.getsize(local_file_path) == 0:
        print(f"Error: {local_file_path} is empty.")
        return

    try:
        s3.upload_file(local_file_path, R2_BUCKET, r2_key)
        print(f"Successfully uploaded {local_file_path} to R2 as {r2_key}")
    except Exception as e:
        print(f"Error uploading {local_file_path}: {str(e)}")

# Connect to MongoDB
from pymongo import MongoClient
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Fetch all documents from the collection
documents = list(collection.find({}))

# Group documents by postid
postid_grouped = {}
for doc in documents:
    postid = doc.get("postid")
    if postid not in postid_grouped:
        postid_grouped[postid] = []
    slug = slugify(doc.get("slug")) if doc.get("slug") else slugify(doc.get("title"))
    postid_grouped[postid].append({
        "_id": str(doc["_id"]),
        "title": doc.get("title"),
        "slug": slug,
        "excerpt": doc.get("excerpt"),
        "outline": doc.get("outline"),
        "exists": True if doc.get("content") else False
    })

# Create local 'feed' folder
os.makedirs("feed", exist_ok=True)

# Save grouped files by postid and upload to Cloudflare R2
for postid, docs in postid_grouped.items():
    local_file_path = f"feed/{postid}.json"
    with open(local_file_path, "w") as f:
        json.dump(docs, f, indent=4)
    upload_to_r2(local_file_path, f"feed/{postid}.json")


# Group documents by category ensuring unique postid per category
category_grouped = {}
for doc in documents:
    category = doc.get("category", "uncategorized").lower()  # Convert category to lowercase
    slugified_category = slugify(category)  # Slugify category
    
    if slugified_category not in category_grouped:
        category_grouped[slugified_category] = {}

    postid = doc.get("postid")
    # If postid not already in the category group, add a random document for that postid
    if postid not in category_grouped[slugified_category]:
        slug = slugify(doc.get("slug")) if doc.get("slug") else slugify(doc.get("title"))
        category_grouped[slugified_category][postid] = {
            "_id": str(doc["_id"]),
            "title": doc.get("title"),
            "slug": slug,
            "excerpt": doc.get("excerpt"),
            "outline": doc.get("outline"),
            "exists": True if doc.get("content") else False
        }

# Shuffle and limit the items to 25, then upload each category file to Cloudflare R2
for category, docs in category_grouped.items():
    unique_docs = list(docs.values())  # Get unique docs based on postid
    # if less then 10 skip the category
    if len(unique_docs) < 10:
        print(f"Skipping {category} as it has less than 10 unique documents.")
        continue
    random.shuffle(unique_docs)
    limited_docs = unique_docs[:100]
    
    local_file_path = f"feed/{category}.json"
    with open(local_file_path, "w") as f:
        json.dump(limited_docs, f, indent=4)
    upload_to_r2(local_file_path, f"feed/{category}.json")

# clean up - delete all files in the feed folder
for file in os.listdir("feed"):
    os.remove(os.path.join("feed", file))

print("Files have been created and uploaded to Cloudflare R2 successfully.")

Uploading feed/None.json to R2 as feed/None.json
Successfully uploaded feed/None.json to R2 as feed/None.json
Uploading feed/54f146bf29ee86141a496042.json to R2 as feed/54f146bf29ee86141a496042.json
Successfully uploaded feed/54f146bf29ee86141a496042.json to R2 as feed/54f146bf29ee86141a496042.json
Uploading feed/54252fec63d106f9599e06dd.json to R2 as feed/54252fec63d106f9599e06dd.json
Successfully uploaded feed/54252fec63d106f9599e06dd.json to R2 as feed/54252fec63d106f9599e06dd.json
Uploading feed/53655675aab5a3eb796c927a.json to R2 as feed/53655675aab5a3eb796c927a.json
Successfully uploaded feed/53655675aab5a3eb796c927a.json to R2 as feed/53655675aab5a3eb796c927a.json
Uploading feed/52472938d65908c77cf3d09d.json to R2 as feed/52472938d65908c77cf3d09d.json
Successfully uploaded feed/52472938d65908c77cf3d09d.json to R2 as feed/52472938d65908c77cf3d09d.json
Uploading feed/53db54cec6a8a1ef0f0def79.json to R2 as feed/53db54cec6a8a1ef0f0def79.json
Successfully uploaded feed/53db54cec6a8a1